In [109]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time

In [110]:
# Settings
Base_url = "https://books.toscrape.com/"

In [111]:
data = r.get(Base_url).text

In [112]:
soup = bs(data, "html.parser")

In [113]:
# Title
print(soup.title.text)


    All products | Books to Scrape - Sandbox



In [114]:
# Body
print(soup.find("h1"))

<h1>All products</h1>


In [137]:
# HTML elements

books =soup.find("article", class_="product_pod")

In [138]:
print(books.prettify())

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [118]:
title = book.h3.a["title"]
print(title)

A Light in the Attic


In [119]:
availability = book.find('p', class_='instock').text.strip()
print(availability)

In stock


In [131]:
rating = book.find("p", class_="star-rating")['class'][1]
print(rating)

In [149]:
book.select_one(".price_color").get_text(strip=True)

'Â£51.77'

In [122]:
book_url = book.h3.a['href']
print(book_url)

catalogue/a-light-in-the-attic_1000/index.html


In [ ]:
book.find("P", class_='price-color')

In [153]:
url = "https://books.toscrape.com/"
response = r.get(url)
soup = bs(response.text,"html.parser")

books = soup.find_all('article', class_="product_pod")
class_ = "product_pod"

book_data = []

for book in books:
    title = book.h3.a["title"]
    price = book.select_one(".price_color").get_text(strip=True)
    rating = book.select_one(".star-rating")["class"][1]
    availability = book.find('p', class_='instock').text.strip()
    url = book.h3.a['href']

    book_data.append({
        "Title" : title,
        "Price" : price,
        "Rating" : rating,
        "Availability" : availability,
        "URL" : url
    })

df = pd.DataFrame(book_data)
print(df.head())

                                   Title    Price Rating Availability  \
0                   A Light in the Attic  Â£51.77  Three     In stock   
1                     Tipping the Velvet  Â£53.74    One     In stock   
2                             Soumission  Â£50.10    One     In stock   
3                          Sharp Objects  Â£47.82   Four     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23   Five     In stock   

                                                 URL  
0     catalogue/a-light-in-the-attic_1000/index.html  
1        catalogue/tipping-the-velvet_999/index.html  
2                catalogue/soumission_998/index.html  
3             catalogue/sharp-objects_997/index.html  
4  catalogue/sapiens-a-brief-history-of-humankind...  


In [154]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Title         20 non-null     object
 1   Price         20 non-null     object
 2   Rating        20 non-null     object
 3   Availability  20 non-null     object
 4   URL           20 non-null     object
dtypes: object(5)
memory usage: 928.0+ bytes


# Create a reusable Scraping function

In [165]:
base_url = 'https://books.toscrape.com/'

def scrape_page(url):
    response = requests.get(url)

    if response.status_code != 200:
        print("Failed to access:", url)
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_='product_pod')

    results = []

    for book in books:
        title = book.h3.a['title']

        price = book.select_one(".price_color").get_text(strip=True)

        rating = book.select_one(".star-rating")["class"][1]
        availability = book.find('p', class_='instock').text.strip()
        relative_url = book.h3.a['href']
        product_url = urljoin(url, relative_url)

        results.append({
                
            "Title" : title,
            "Price" : price,
            "Rating" : rating,
            "Availability" : availability,
            "URL" : product_url            
        })
    return results

In [168]:
# Test it on page 1
data = scrape_page(base_url)

print("Books Collected:", len(data))
print(data[:1])

Books Collected: 20
[{'Title': 'A Light in the Attic', 'Price': 'Â£51.77', 'Rating': 'Three', 'Availability': 'In stock', 'URL': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}]


# Scrape multiple pages

Now comes the important part: web navigation.

In [171]:
all_books = []

for page in range(1,51):
    if page ==1:
        url = base_url
    else:
        url = urljoin(
            base_url,
            f"catalogue/page-{page}.html"
        )
    print(f"Scraping Page {page}...")

    page_data = scrape_page(url)

    if not page_data:
        print("No more data found.")
        break

    all_books.extend(page_data)

    time.sleep(1)

print("Total books Collected: ", len(all_books))

Scraping Page 1...
Scraping Page 2...
Scraping Page 3...
Scraping Page 4...
Scraping Page 5...
Scraping Page 6...
Scraping Page 7...
Scraping Page 8...
Scraping Page 9...
Scraping Page 10...
Scraping Page 11...
Scraping Page 12...
Scraping Page 13...
Scraping Page 14...
Scraping Page 15...
Scraping Page 16...
Scraping Page 17...
Scraping Page 18...
Scraping Page 19...
Scraping Page 20...
Scraping Page 21...
Scraping Page 22...
Scraping Page 23...
Scraping Page 24...
Scraping Page 25...
Scraping Page 26...
Scraping Page 27...
Scraping Page 28...
Scraping Page 29...
Scraping Page 30...
Scraping Page 31...
Scraping Page 32...
Scraping Page 33...
Scraping Page 34...
Scraping Page 35...
Scraping Page 36...
Scraping Page 37...
Scraping Page 38...
Scraping Page 39...
Scraping Page 40...
Scraping Page 41...
Scraping Page 42...
Scraping Page 43...
Scraping Page 44...
Scraping Page 45...
Scraping Page 46...
Scraping Page 47...
Scraping Page 48...
Scraping Page 49...
Scraping Page 50...
Total boo

In [179]:
# Create DataFrame
df = pd.DataFrame(all_books)
df.head()

,Title,Price,Rating,Availability,URL
0,A Light in the Attic,Â£51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Â£53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Â£50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Â£47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...


In [181]:
df.shape

(1000, 5)

In [182]:
df['Price'] = (df["Price"]
              .str.replace("Â£", "", regex=False)
              .astype(float))

In [192]:
# Convert ratings

rating_map = {
    "One" : 1,
    "Two" : 2,
    "Three" : 3,
    "Four" : 4,
    "Five" : 5
    
}
df["Rating "] = df["Rating"].map(rating_map)

In [193]:
df.head()

,Title,Price,Rating,Availability,URL,Rating
0,A Light in the Attic,51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...,3
1,Tipping the Velvet,53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...,1
2,Soumission,50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...,1
3,Sharp Objects,47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...,4
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...,5


In [201]:
df = df.drop(columns=['Rating'])

In [202]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Title         1000 non-null   object 
 1   Price         1000 non-null   float64
 2   Availability  1000 non-null   object 
 3   URL           1000 non-null   object 
 4   Rating        1000 non-null   int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 39.2+ KB


# Perform data-quality checks

In [203]:
# Check ratings

print(df["Price"].describe())

count    1000.00000
mean       35.07035
std        14.44669
min        10.00000
25%        22.10750
50%        35.98000
75%        47.45750
max        59.99000
Name: Price, dtype: float64


In [187]:
df.isnull().sum()

Title           0
Price           0
Rating          0
Availability    0
URL             0
Rating          0
dtype: int64

In [190]:
# Check ratings:

print(df["Rating"].value_counts().sort_index())

Rating
Five     196
Four     179
One      226
Three    203
Two      196
Name: count, dtype: int64


In [204]:
# Save the final dataset

df.to_csv(
    "scraped_books_dataset.csv",
    index=False
)

In [205]:
df1 = pd.read_csv("scraped_books_dataset.csv")

In [206]:
df1.head()

,Title,Price,Availability,URL,Rating
0,A Light in the Attic,51.77,In stock,https://books.toscrape.com/catalogue/a-light-i...,3
1,Tipping the Velvet,53.74,In stock,https://books.toscrape.com/catalogue/tipping-t...,1
2,Soumission,50.10,In stock,https://books.toscrape.com/catalogue/soumissio...,1
3,Sharp Objects,47.82,In stock,https://books.toscrape.com/catalogue/sharp-obj...,4
4,Sapiens: A Brief History of Humankind,54.23,In stock,https://books.toscrape.com/catalogue/sapiens-a...,5
